In [0]:
STORAGE_ACCOUNT = 'stccasemauricioes'
CONTAINER = 'data'
STORAGE_KEY = 'COLOCAR_CHAVE_AQUI'

spark.conf.set("fs.azure.account.key." + STORAGE_ACCOUNT + ".blob.core.windows.net", STORAGE_KEY)

BASE_BRONZE = "wasbs://" + CONTAINER + "@" + STORAGE_ACCOUNT + ".blob.core.windows.net/bronze"
BASE_SILVER = "wasbs://" + CONTAINER + "@" + STORAGE_ACCOUNT + ".blob.core.windows.net/silver"

In [0]:
"""
Pipeline Medallion - Camada SILVER
Tipagem, dedup, quality gates.
"""
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# === PEDIDOS ===
print('Processando pedidos...')
bronze_pedidos = spark.read.format('delta').load(f'{BASE_BRONZE}/pedidos')
n_bronze = bronze_pedidos.count()
print(f'  Bronze: {n_bronze:,}')

# Tipagem
pedidos_tipados = (bronze_pedidos
    .withColumn('subtotal',     F.col('subtotal').cast('decimal(10,2)'))
    .withColumn('taxa_entrega', F.col('taxa_entrega').cast('decimal(10,2)'))
    .withColumn('total',        F.col('total').cast('decimal(10,2)'))
    .withColumn('criado_em_clean', F.regexp_replace(F.col('criado_em'), 'Z$', ''))
    .withColumn('criado_em',    F.to_timestamp('criado_em_clean'))
    .drop('criado_em_clean'))

# Dedup - mantem versao mais recente
w_dedup = Window.partitionBy('pedido_id').orderBy(F.desc('_ingestion_timestamp'))
pedidos_dedup = (pedidos_tipados
    .withColumn('_rn', F.row_number().over(w_dedup))
    .filter(F.col('_rn') == 1)
    .drop('_rn'))

n_apos = pedidos_dedup.count()
print(f'  Apos dedup: {n_apos:,} (removidos {n_bronze - n_apos} duplicados)')

# Quality gates
pedidos_validos = pedidos_dedup.filter(
    (F.col('pedido_id').isNotNull()) &
    (F.col('cliente_id').isNotNull()) &
    (F.col('loja_id').isNotNull()) &
    (F.col('total') > 0) &
    (F.col('status').isin('PAGO', 'PENDENTE', 'CANCELADO')) &
    (F.col('criado_em').isNotNull()))

pedidos_rejeitados = pedidos_dedup.exceptAll(pedidos_validos)
n_rej = pedidos_rejeitados.count()
print(f'  Rejeitados: {n_rej}')

if n_rej > 0:
    (pedidos_rejeitados
        .withColumn('_rejection_reason', F.lit('falhou_quality_gate'))
        .write.mode('append').format('delta')
        .save(f'{BASE_SILVER}/_rejected/pedidos'))

# MERGE upsert
silver_path = f'{BASE_SILVER}/pedidos'
if DeltaTable.isDeltaTable(spark, silver_path):
    silver = DeltaTable.forPath(spark, silver_path)
    (silver.alias('tgt')
        .merge(pedidos_validos.alias('src'), 'tgt.pedido_id = src.pedido_id')
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    print('  MERGE executado')
else:
    pedidos_validos.write.format('delta').mode('overwrite').save(silver_path)
    print('  Tabela criada')

Processando pedidos...
  Bronze: 4,946
  Apos dedup: 4,896 (removidos 50 duplicados)
  Rejeitados: 0
  Tabela criada


In [0]:
# === PEDIDO_ITENS ===
print('\nProcessando itens...')
bronze_itens = spark.read.format('delta').load(f'{BASE_BRONZE}/pedido_itens')

itens_tipados = (bronze_itens
    .withColumn('quantidade',     F.col('quantidade').cast('int'))
    .withColumn('preco_unitario', F.col('preco_unitario').cast('decimal(10,2)'))
    .withColumn('subtotal',       F.col('subtotal').cast('decimal(10,2)')))

w_it = Window.partitionBy('item_id').orderBy(F.desc('_ingestion_timestamp'))
itens_dedup = (itens_tipados
    .withColumn('_rn', F.row_number().over(w_it))
    .filter(F.col('_rn') == 1).drop('_rn'))

itens_validos = itens_dedup.filter(
    (F.col('item_id').isNotNull()) &
    (F.col('pedido_id').isNotNull()) &
    (F.col('quantidade') > 0) &
    (F.col('subtotal') > 0))

n_rej_it = itens_dedup.count() - itens_validos.count()
print(f'  Itens rejeitados (subtotal zerado): {n_rej_it}')

silver_path = f'{BASE_SILVER}/pedido_itens'
if DeltaTable.isDeltaTable(spark, silver_path):
    silver = DeltaTable.forPath(spark, silver_path)
    (silver.alias('tgt')
        .merge(itens_validos.alias('src'), 'tgt.item_id = src.item_id')
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
else:
    itens_validos.write.format('delta').mode('overwrite').save(silver_path)
print(f'  Silver itens: {itens_validos.count():,}')

# === PAGAMENTOS ===
print('\nProcessando pagamentos...')
bronze_pag = spark.read.format('delta').load(f'{BASE_BRONZE}/pagamentos')

pag_tipados = (bronze_pag
    .withColumn('valor', F.col('valor').cast('decimal(10,2)'))
    .withColumn('processado_em', F.to_timestamp('processado_em')))

w_pag = Window.partitionBy('pagamento_id').orderBy(F.desc('_ingestion_timestamp'))
pag_validos = (pag_tipados
    .withColumn('_rn', F.row_number().over(w_pag))
    .filter(F.col('_rn') == 1).drop('_rn')
    .filter(F.col('valor') > 0))

silver_path = f'{BASE_SILVER}/pagamentos'
if DeltaTable.isDeltaTable(spark, silver_path):
    silver = DeltaTable.forPath(spark, silver_path)
    (silver.alias('tgt')
        .merge(pag_validos.alias('src'), 'tgt.pagamento_id = src.pagamento_id')
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
else:
    pag_validos.write.format('delta').mode('overwrite').save(silver_path)
print(f'  Silver pagamentos: {pag_validos.count():,}')


Processando itens...
  Itens rejeitados (subtotal zerado): 20
  Silver itens: 12,098

Processando pagamentos...
  Silver pagamentos: 4,623


In [0]:
# === DIMENSOES (produtos, lojas, clientes) ===
print('\nProcessando dimensoes...')
for tabela in ['produtos', 'lojas', 'clientes']:
    df = (spark.read.format('delta').load(f'{BASE_BRONZE}/{tabela}')
          .withColumn('_silver_updated_at', F.current_timestamp()))
    if tabela == 'produtos':
        df = (df.withColumn('preco_base', F.col('preco_base').cast('decimal(10,2)'))
                .withColumn('abv', F.col('abv').cast('decimal(4,2)')))
    df.write.format('delta').mode('overwrite').save(f'{BASE_SILVER}/{tabela}')
    print(f'  {tabela}: {df.count():,}')

print('\n=== SILVER OK ===')


Processando dimensoes...
  produtos: 15
  lojas: 12
  clientes: 2,000

=== SILVER OK ===
